# sample_Create_Doc

https://github.com/PacktPublishing/Building-Data-Driven-Applications-with-LlamaIndex/tree/main/ch3

In [1]:
from llama_index.core import Document

text = "The quick brown fox jumps over the lazy dog."
doc = Document(
    text=text, 
    metadata={'author': 'John Doe','category': 'others'}, 
    id_='1'
)
print(doc)

Doc ID: 1
Text: The quick brown fox jumps over the lazy dog.


```
pip install wikipedia
pip install llama-index-readers-wikipedia
```

In [ ]:
from llama_index.readers.wikipedia import WikipediaReader
loader = WikipediaReader()
documents = loader.load_data(
    pages=['Pythagorean theorem','General relativity']
)
print(f"loaded {len(documents)} documents")

# creating the Node objects

In [2]:
from llama_index.core import Document
from llama_index.core.schema import TextNode
doc = Document(text="This is a sample document text")
n1 = TextNode(text=doc.text[0:16], doc_id=doc.id_)
n2 = TextNode(text=doc.text[17:30], doc_id=doc.id_)
print(n1)
print(n2)

Node ID: b5276f14-c858-45da-9085-c36b1291feb3
Text: This is a sample
Node ID: d552fb24-9b08-42f5-991e-cd04eb63a29c
Text: document text


# sample_TokenTextSplitter.

In [1]:
from llama_index.core import Document
from llama_index.core.node_parser import TokenTextSplitter

doc = Document( 
    text=(
    "This is sentence 1. This is sentence 2. "
    "Sentence 3 here."
    ),
    metadata={"author": "John Smith"}
)  
splitter = TokenTextSplitter( 
    chunk_size=12, 
    chunk_overlap=0, 
    separator=" "
) 

nodes = splitter.get_nodes_from_documents([doc]) 
for node in nodes: 
    print(node.text) 
    print(node.metadata)

Metadata length (6) is close to chunk size (12). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.
This is sentence 1.
{'author': 'John Smith'}
This is sentence 2.
{'author': 'John Smith'}
Sentence 3 here.
{'author': 'John Smith'}


### Here’s an example that manually creates a simple relationship between two Nodes:

In [1]:
from llama_index.core import Document
from llama_index.core.schema import (
    TextNode, 
    NodeRelationship, 
    RelatedNodeInfo
)
doc = Document(text="First sentence. Second Sentence")
n1 = TextNode(text="First sentence", node_id=doc.doc_id)
n2 = TextNode(text="Second sentence", node_id=doc.doc_id)

n1.relationships[NodeRelationship.NEXT] = n2.node_id 
n2.relationships[NodeRelationship.PREVIOUS] = n1.node_id
print(n1.relationships)
print(n2.relationships)


{<NodeRelationship.NEXT: '3'>: 'd7b74934-74b3-4b04-8400-e0c8eafb374e'}
{<NodeRelationship.PREVIOUS: '2'>: 'cb631643-e46b-4a0b-9850-54040d3a1578'}


# sample_Create_and_Query_Index.py

In [ ]:
from llama_index.core import SummaryIndex, Document
from llama_index.core.schema import TextNode

nodes = [
  TextNode(
    text="Lionel Messi is a football player from Argentina."
    ),
  TextNode(
    text="He has won the Ballon d'Or trophy 7 times."
    ),
  TextNode(text="Lionel Messi's hometown is Rosario."),
  TextNode(text="He was born on June 24, 1987.")
]
index = SummaryIndex(nodes)

query_engine = index.as_query_engine()
response = query_engine.query(
    "What is Messi's hometown?"
)
print(response)  # WTF No necesita de una llm !!!!

Rosario


In [6]:
import os
key = os.environ.get("OPENAI_API_KEY")
if key:
    print(f"¡Encontré una API Key!: {key[:5]}...")
else:
    print("No hay API Key configurada.")

¡Encontré una API Key!: sk-pr...


#### Building our first interactive, augmented LLM application

In [ ]:
from llama_index.core import Document, SummaryIndex
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.readers.wikipedia import WikipediaReader

loader = WikipediaReader()
documents = loader.load_data(pages=["Messi Lionel"])
parser = SimpleNodeParser.from_defaults()
nodes = parser.get_nodes_from_documents(documents)
index = SummaryIndex(nodes)
query_engine = index.as_query_engine()
print("Ask me anything about Lionel Messi!")

while True:
    question = input("Your question: ")
    if question.lower() == "exit":
        break
    response = query_engine.query(question)
    print(response)